In [ ]:
from torchvision.datasets import CIFAR10
from torchvision.transforms.functional import to_tensor
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch
from torch.optim import AdamW
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder,StandardScaler


In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)
print("total misisng values", df.isnull().sum().sum())
#fill missing values with the mean
for col in df.columns:
  df[col] = df[col].fillna(df[col].mean())
check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
print(categorical_cols)
#no need

In [ ]:
# Task 4: Write your code here:
numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Target")

scaler = StandardScaler()

# scale the `numerical_cols`
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.head()


In [ ]:
# Task 5: Write your code here:
percent_of_one = len(df[(df['Target'] == 1)])/len(df['Target'])
print(f"percent of Target that equals 1: {percent_of_one*100:.2f}%")
#Target = 1 is the minority class

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q

clear_output()

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier
F1 = []
acc = []
n_splits = 3 # K=3 Folds
model = CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
sr_results = {'loss': [], 'acc': [], 'f1': []}
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train using gradient descent with learning rate = 0.5
  model.fit(X_train, y_train)

  # Calculate z & class probabilities for X_test
  y_pred = model.predict(X_test)

    # Calculate evaluation metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

  acc.append(accuracy)
  F1.append(f1)

In [ ]:
# Task 1: Write your code here:
importances = {}

importances['CatBoost'] = model.feature_importances_
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()
# Create a 1x3 plot
sorted_idx = np.argsort(importances['CatBoost'][:5])
features = X.columns

plt.tight_layout()
plt.show()

In [ ]:
sort_idx = np.argsort(model.feature_importances_)
print(features[sort_idx][:5])
plt.barh(features[sort_idx][:8],model.feature_importances_[:8])

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: